# Pandas Lab — AI Lab Instructor Notebook

*Python & Data · Easy*

Build strong intuition for Pandas transformations used in real ML pipelines (feature engineering, joins, groupby, leakage-safe logic).

**Outcome.** Students can write correct, efficient Pandas code, explain tradeoffs, and avoid common interview/production gotchas.

8 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import pandas as pd
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

## Task 1: Dataset & Setup

# Pandas Lab — Student Notebook

Complete each `# TODO` cell. Run the checks to verify your work.
Use `Shift+Enter` to run each cell.

In [ ]:
import pandas as pd
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)
users = pd.DataFrame({
    'user_id': [101, 102, 103, 104],
    'country': ['IN', 'US', 'IN', 'CA'],
    'signup_ts': pd.to_datetime(['2025-01-01', '2025-01-03', '2025-01-04', '2025-01-10'])
})

events = pd.DataFrame({
    'user_id': rng.choice(users['user_id'], size=30, replace=True),
    'event_ts': pd.to_datetime('2025-01-01') + pd.to_timedelta(rng.integers(0, 20*24*60, size=30), unit='m'),
    'event': rng.choice(['view', 'add_to_cart', 'purchase'], size=30, replace=True, p=[0.7, 0.2, 0.1]),
    'amount': rng.choice([0, 0, 0, 19.99, 49.99, 99.99], size=30, replace=True),
})
events.loc[events['event'] != 'purchase', 'amount'] = 0
print(users.head(), events.head())

**Why this works.** # Pandas Lab — Instructor Notebook (Solutions + Rationale)

Complete solutions with short rationale. Runs top-to-bottom.

## Task 2: Filter purchases and extract event day
*Section: DataFrame Fundamentals*

## Section 1 — DataFrame Fundamentals

### Task 1.1: Filter purchases and extract event day

Filter `events` to only purchase rows, then create an `event_day` column by flooring `event_ts` to day.

**HINT:** Use `events[events['event'] == 'purchase']` and `.dt.floor('D')`

In [ ]:
purchases = events[events['event'] == 'purchase'].copy()
purchases['event_day'] = purchases['event_ts'].dt.floor('D')

check('purchases_only', set(purchases['event'].unique()) <= {'purchase'})
check('event_day_dtype', str(purchases['event_day'].dtype).startswith('datetime'))

# Rationale: `.dt` operations are vectorized in C/numexpr-like paths;
# per-row parsing introduces Python overhead and is slower / more error-prone.

In [ ]:
# Checks
check('purchases_only', set(purchases['event'].unique()) <= {'purchase'})
check('event_day_dtype', str(purchases['event_day'].dtype).startswith('datetime'))

**Why this works.** `.dt` operations are vectorized in C/numexpr-like paths;

## Task 3: Build user-level features with groupby
*Section: GroupBy*

## Section 2 — GroupBy

### Task 2.1: Build user-level features with groupby

Group events by `user_id` and compute:
- `n_events`: count of events
- `n_purchases`: sum of purchase flags
- `total_revenue`: sum of amount

**HINT:** Use `.assign()` to create `is_purchase`, then `.groupby().agg()` with named aggregations

In [ ]:
user_features = (
    events.assign(is_purchase=(events['event'] == 'purchase').astype(int))
    .groupby('user_id', as_index=True)
    .agg(
        n_events=('event', 'size'),
        n_purchases=('is_purchase', 'sum'),
        total_revenue=('amount', 'sum'),
    )
)
user_features = user_features.reset_index()
print(user_features)
check('has_user_id', 'user_id' in user_features.columns)
check('has_total_revenue', 'total_revenue' in user_features.columns)

# Rationale: one groupby with named aggregations is idiomatic and avoids multiple passes.

In [ ]:
# Checks
check('has_user_id', 'user_id' in user_features.columns)
check('has_total_revenue', 'total_revenue' in user_features.columns)

**Why this works.** one groupby with named aggregations is idiomatic and avoids multiple passes.

## Task 4: Row-level feature with transform
*Section: GroupBy*

### Task 2.2: Row-level feature with transform

Add a `user_event_count` column to events using `transform('size')` — this keeps the original row count.

**HINT:** `groupby('user_id')['event'].transform('size')`

In [ ]:
events2 = events.copy()
events2['user_event_count'] = events2.groupby('user_id')['event'].transform('size')
check('user_event_count_nonnull', events2['user_event_count'].notna().all())
check('counts_positive', (events2['user_event_count'] > 0).all())

# Rationale: transform returns a vector aligned to the original rows (row-level feature),
# whereas agg returns one row per group (group-level feature).

In [ ]:
# Checks
check('user_event_count_nonnull', events2['user_event_count'].notna().all())
check('counts_positive', (events2['user_event_count'] > 0).all())

**Why this works.** transform returns a vector aligned to the original rows (row-level feature),

## Task 5: Left-join users with user_features
*Section: Joins & Merge*

## Section 3 — Joins & Merge

### Task 3.1: Left-join users with user_features

Merge `users` with `user_features` on `user_id` (left join). Fill NaN numeric columns with 0.

**HINT:** `users.merge(user_features, on='user_id', how='left')` then `.fillna(0)`

In [ ]:
user_table = users.merge(user_features, on='user_id', how='left')
for col in ['n_events', 'n_purchases', 'total_revenue']:
    user_table[col] = user_table[col].fillna(0)

check('rowcount_preserved', len(user_table) == len(users))
user_table

# Rationale: Always validate row counts after joins to catch many-to-many explosions early.

In [ ]:
# Checks
check('rowcount_preserved', len(user_table) == len(users))

**Why this works.** Always validate row counts after joins to catch many-to-many explosions early.

## Task 6: Diagnose and fix a many-to-many join explosion
*Section: Joins & Merge*

### Task 3.2: Diagnose and fix a many-to-many join explosion

Join `left` and `right` on key `k`. Notice the row explosion. Fix it by deduplicating the right side first.

**FAANG gotcha:** Always validate row counts after joins to catch many-to-many explosions early.

In [ ]:
left = pd.DataFrame({'k': [1, 1, 2], 'l': ['a', 'b', 'c']})
right = pd.DataFrame({'k': [1, 1, 2], 'r': ['x', 'y', 'z']})
exploded = left.merge(right, on='k', how='inner')
print('left rows', len(left), 'right rows', len(right), 'merged rows', len(exploded))

right_dedup = right.drop_duplicates('k', keep='first')
fixed = left.merge(right_dedup, on='k', how='inner')
print('fixed rows', len(fixed))

# Rationale: If the downstream expects 1:1 or 1:n, you must enforce unique keys
# (dedupe / aggregate) on the appropriate side.

**Why this works.** If the downstream expects 1:1 or 1:n, you must enforce unique keys

## Task 7: Cumulative purchase count (strictly before current row)
*Section: Leakage-safe Time Features*

## Section 4 — Leakage-safe Time Features

### Task 4.1: Cumulative purchase count (strictly before current row)

Sort events by user and time, compute cumulative purchase count, then shift by 1 so each row only sees past data.

**FAANG gotcha:** Using `cumsum()` without `shift(1)` leaks the current row's label into the feature.

**HINT:** `groupby().cumsum()` then `groupby().shift(1).fillna(0)`

In [ ]:
events3 = events.copy().sort_values(['user_id', 'event_ts']).reset_index(drop=True)
events3['is_purchase'] = (events3['event'] == 'purchase').astype(int)
events3['purchases_cum'] = events3.groupby('user_id')['is_purchase'].cumsum()
events3['purchases_before'] = events3.groupby('user_id')['purchases_cum'].shift(1).fillna(0).astype(int)

check('nonnegative', (events3['purchases_before'] >= 0).all())
print(events3[['user_id','event_ts','event','purchases_before']].head(10))

# Rationale: shift(1) makes it strictly-before and prevents label leakage for purchase rows.

In [ ]:
# Checks
check('nonnegative', (events3['purchases_before'] >= 0).all())

**Why this works.** shift(1) makes it strictly-before and prevents label leakage for purchase rows.

## Task 8: Compare apply vs vectorized boolean
*Section: Apply vs Vectorization*

## Section 5 — Apply vs Vectorization

### Task 5.1: Compare apply vs vectorized boolean

Create `country_is_in` column two ways: with `.apply(lambda)` and with vectorized `==`. Compare timing.

Use a larger DataFrame so timing is meaningful.

**HINT:**
- Build `u` with many rows using `pd.concat([users]*25000, ignore_index=True)`
- Vectorized: `u['country'] == 'IN'`

In [ ]:
import time
u = pd.concat([users] * 25000, ignore_index=True)  # 100k rows

t1 = time.perf_counter()
u['country_is_in_apply'] = u['country'].apply(lambda x: x == 'IN')
t_apply = time.perf_counter() - t1

t2 = time.perf_counter()
u['country_is_in'] = u['country'] == 'IN'
t_vectorized = time.perf_counter() - t2

check('same_result', (u['country_is_in_apply'] == u['country_is_in']).all())
check('large_enough', len(u) >= 100_000)
print(u.head())
print('rows:', len(u))
print('apply time:', t_apply)
print('vectorized time:', t_vectorized)
print('speedup (apply/vectorized):', t_apply / max(t_vectorized, 1e-12))

# Rationale: apply calls Python per-row; vectorized ops run in optimized Pandas/NumPy loops.


In [ ]:
# Checks
check('same_result', (u['country_is_in_apply'] == u['country_is_in']).all())
check('large_enough', len(u) >= 100_000)

**Why this works.** apply calls Python per-row; vectorized ops run in optimized Pandas/NumPy loops.